In [14]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [15]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_distances_suffixes.parquet"))

In [16]:
# Council district dummies + some cleaning

vals = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','CITYWIDE']
for v in vals:
    df[f'cd_{v}'] = 0

invalids = []
for idx, row in df.iterrows():
    splitted = re.split(r"[,;]", row['council_district'])
    council_districts = [x.strip() for x in splitted if len(x.strip())>0]
    for cd in council_districts:
        if cd in vals:
            df.loc[idx, f'cd_{cd}'] = 1
        else:
            invalids.append((idx, cd))

print(invalids)

# apply fixes
df.loc[82, "cd_9"] = 1
df.loc[571, "cd_CITYWIDE"] = 1
df.loc[797, "cd_CITYWIDE"] = 1
df.loc[808, "cd_CITYWIDE"] = 1

[(82, 'Multiple'), (571, 'ALL'), (797, 'ALL'), (808, 'ALL')]


In [17]:
# processed variables

# Outcome variable
df["outcome"] = 1
df.loc[df["project_result"] == "APPROVED", "outcome"] = 2
df.loc[df["project_result"].isin(['DELAYED', 'DENIED', 'APPLICATION WIDTHDRAWN']), "outcome"] = 0

# Choose distance measure
df["atypicality"] = df["mahalanobis"]

# project type dummies
df['is_residential'] = df['project_type'].isin(['RESIDENTIAL DEVELOPMENT', 'PERMANENT SUPPORTIVE HOUSING / HOMELESS SHELTER', 'SENIOR CARE / ASSISTED LIVING FACILITY'])
df['is_mixed_use'] = df['project_type'].isin(['MIXED-USE DEVELOPMENT'])
df['is_nonresidential'] = df['project_type'].isin(['NON-RESIDENTIAL DEVELOPMENT'])

# support/opposition letters
df["log2_support"] = np.log2(df["n_support"] + 1)
df["log2_oppose"] = np.log2(df["n_oppose"] + 1)

# height/square footage
df["height"] = pd.to_numeric(df["height"], errors="coerce")
df["square_footage"] = pd.to_numeric(df["square_footage"], errors="coerce")
df["log_square_footage"] = np.log(1 + df["square_footage"])

# handle missing values for height and log_square_footage
df["height_missing"] = df["height"].isna()
df["height"] = df["height"].fillna(0)
df["log_square_footage_missing"] = df["log_square_footage"].isna()
df["log_square_footage"] = df["log_square_footage"].fillna(0)





In [18]:
# Output regression data

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [19]:
# Running R regressions

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/15-ordered-logit.R"], check=True, capture_output=True, text=True)
print(res.stdout)


                                    Dependent variable:          
                           --------------------------------------
                                          outcome                
                              (1)       (2)       (3)      (4)   
-----------------------------------------------------------------
is_residential              0.375*              0.338*   0.412** 
                            (0.195)             (0.188)  (0.191) 
                                                                 
is_nonresidential           -0.207              -0.268    -0.166 
                            (0.258)             (0.237)  (0.251) 
                                                                 
log_square_footage          -0.068              -0.038    -0.078 
                            (0.062)             (0.060)  (0.060) 
                                                                 
log_square_footage_missing  -0.818              -0.410    -0.960 
         

In [20]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_coefs.parquet"))


In [21]:
# Coefficient results

main_reg = 'r1'

AtypicalityCoef = coefs_df.loc[
    (coefs_df['regression_name']==main_reg) & (coefs_df['coef_name']=='atypicality'),
    'estimate'
].values[0]

NumOpposeCoef = coefs_df.loc[
    (coefs_df['regression_name']==main_reg) & (coefs_df['coef_name']=='log2_oppose'),
    'estimate'
].values[0]

ConCalCoef = coefs_df.loc[
    (coefs_df['regression_name']==main_reg) & (coefs_df['coef_name']=='is_consent_calendarTRUE'),
    'estimate'
].values[0]

AgendaOrderCoef = coefs_df.loc[
    (coefs_df['regression_name']==main_reg) & (coefs_df['coef_name']=='agenda_order'),
    'estimate'
].values[0]

RESULTS = {
    "AtypicalityCoef": f"{np.abs(AtypicalityCoef):.3f}",
    "NumOpposeCoef": f"{np.abs(NumOpposeCoef):.3f}",
    "ConCalCoef": f"{np.abs(ConCalCoef):.3f}",
    "AgendaOrderCoef": f"{np.abs(AgendaOrderCoef):.3f}",
}

_ = wt.update_results(RESULTS)

RESULTS

{'AtypicalityCoef': '0.226',
 'NumOpposeCoef': '0.148',
 'ConCalCoef': '1.649',
 'AgendaOrderCoef': '0.122'}

In [23]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Ordered Logit Regression Results}
\vspace{0.2cm}
\label{tab_ologit_results}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the ordered logit regression described in Section \ref{sec_methodology}.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1", "r2", "r3","r4"]

vars = [
    ("atypicality", "Case Atypicality"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("is_residentialTRUE", "Residential Development"),
    ("is_nonresidentialTRUE", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "\n & & & & \\\\ \n"

tbl += "Embedding Cluster FE "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cluster_fe1TRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Case Suffix Group FE "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="sfx_grp_CUPTRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Council District FE "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cd_1")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "\n & & & & \\\\ \n"

vars = [
    ("0|1", "$\\mu_0$"),
    ("1|2", "$\\mu_1$")
]

for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_ologit_results.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Ordered Logit Regression Results}
\vspace{0.2cm}
\label{tab_ologit_results}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
Case Atypicality  & -0.226$^{**}$ & -0.218$^{**}$ & -0.219$^{**}$ & -0.211$^{**}$ \\
 & (0.106) & (0.105) & (0.100) & (0.103) \\ [1.8ex]
Agenda Order  & 0.122$^{**}$ & 0.127$^{***}$ & 0.125$^{***}$ & 0.124$^{***}$ \\
 & (0.049) & (0.048) & (0.047) & (0.048) \\ [1.8ex]
No. Agenda Items  & -0.036$^{}$ & -0.038$^{}$ & -0.040$^{}$ & -0.046$^{}$ \\
 & (0.040) & (0.040) & (0.039) & (0.039) \\ [1.8ex]
Consent Calendar  & 1.649$^{***}$ & 1.603$^{***}$ & 1.543$^{***}$ & 1.645$^{***}$ \\
 & (0.261) & (0.258) & (0.253) & (0.256) \\ [1.8ex]
$\log_2$(\# Support)  & 0.053$^{}$ & 0.043$^{}$ & 0.061$^{}$ & 0.049$^{}$ \\
 & (0.061) & (0.060) & (0.060) & (0.059) \\ [1.8ex]
$\log_2$(\# Oppose)  & -0.148$^{***}$ & -0.153$^{***}$ & -0.142$^{***

In [ ]:
# Load marginals

marginals_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_marginals.parquet"))

In [ ]:
# Marginal effects results

_results = {}
for term in ["atypicality", "log2_oppose", "is_consent_calendar"]:
    for group in ["0", "1", "2"]:
        mask = (marginals_df['regression_name']=='r4') & (marginals_df['term']==term) & (marginals_df['group']==group)
        _results[(term, group)] = marginals_df.loc[mask, 'estimate'].values[0]

RESULTS = {
    "SemUniME2": f"{np.abs(_results[('atypicality', '2')])*100:.1f}",
    "SemUniME1": f"{np.abs(_results[('atypicality', '1')])*100:.1f}",
    "SemUniME0": f"{np.abs(_results[('atypicality', '0')])*100:.1f}",
    "NOppME2": f"{np.abs(_results[('log2_oppose', '2')])*100:.1f}",
    "NOppME1": f"{np.abs(_results[('log2_oppose', '1')])*100:.1f}",
    "NOppME0": f"{np.abs(_results[('log2_oppose', '0')])*100:.1f}",
     "ConCalME2": f"{np.abs(_results[('is_consent_calendar', '2')])*100:.1f}",
     "ConCalME1": f"{np.abs(_results[('is_consent_calendar', '1')])*100:.1f}",
     "ConCalME0": f"{np.abs(_results[('is_consent_calendar', '0')])*100:.1f}",
}

_ = wt.update_results(RESULTS)

RESULTS

In [ ]:
# Marginal effects table

header = r"""\begin{table}[H]
\centering
\caption{Ordered Logit Marginal Effects}
\vspace{0.2cm}
\label{tab_ologit_marginal_effects}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lccc}
\toprule
 & (1) & (2) & (3) \\
\midrule
 & Outcome 0 & Outcome 1 & Outcome 2 \\
 &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports estimated marginal effects from the ordered logit regression in column (4) of Table \ref{tab_ologit_results}.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
outcomes = ["0", "1", "2"]

vars = [
    ("atypicality", "Case Atypicality"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendar", "Consent Calendar"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for y in outcomes:
        idx = (marginals_df["regression_name"]=="r4") & (marginals_df["term"]==v[0]) & (marginals_df["group"]==y)
        if idx.sum()==0:
            tbl += " & "
            continue
        meff = marginals_df.loc[idx, "estimate"].values[0]
        serr = marginals_df.loc[idx, "std.error"].values[0]
        stars = utils.stars(meff, serr)
        tbl += f" & {meff:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for y in outcomes:
        idx = (marginals_df["regression_name"]=="r4") & (marginals_df["term"]==v[0]) & (marginals_df["group"]==y)
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = marginals_df.loc[idx, "std.error"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += r"""& & & \\
Embedding Cluster FE         & Y & Y & Y \\
Council District FE         & Y & Y & Y \\
Case Suffix Group FE        & Y & Y & Y \\
& & & \\
"""

tbl += "Observations "
for y in outcomes:
    nobs = (df["outcome"]==int(y)).sum()
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_ologit_marginal_effects.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


In [ ]:
# Oster (2019) delta*

oster_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_oster_delta.parquet"))
delta_star = oster_df.loc[0, "Value"]

RESULTS = {
    "OsterDelta": f"{delta_star:.2f}"
}

_ = wt.update_results(RESULTS)

RESULTS

In [ ]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_coefs_nodelay.parquet"))


In [ ]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Binary Logit Regression Results - Non-Delay Outcomes}
\vspace{0.2cm}
\label{tab_blogit_results}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  & \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the binary logit regression described in Section \ref{sec_results}, in which cases where delay was the outcome are removed from the data. The outcome is 1 if the case is approved, and 0 if the case is approved with conditions or denied.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1b", "r2b", "r3b","r4b"]

vars = [
    ("atypicality", "Case Atypicality"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += r"""& & & & \\
Embedding Cluster FE         & N & Y & Y & Y \\
Council District FE         & N & N & Y & Y \\
Case Suffix Group FE        & N & N & N & Y \\
& & & & \\
"""

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_blogit_results.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)
